
# Marine Drifter Forecasting with Attention Transformer + Multi-Step Neural ODE

Features:
- Attention Pooling Transformer
- Multi-Step Neural ODE
- Physics Residual Learning
- Multi-Horizon Forecasting
- Residual Normalization
- Haversine Evaluation
- Checkpoint Resume Support


## Imports

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install torchdiffeq
import os
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import mean_squared_error

from torchdiffeq import odeint

from tqdm import tqdm


## Load Dataset

In [ ]:

def load_year_npz(npz_path):

    data = np.load(npz_path, allow_pickle=True)

    return {
        "X_train": data["X_train"],
        "y_train": data["y_train"],

        "X_val": data["X_val"],
        "y_val": data["y_val"],

        "X_test": data["X_test"],
        "y_test": data["y_test"],

        "feature_cols": data["feature_cols"]
    }

base_dir = "/content/drive/MyDrive/2022-2025_processed_data_extended"

years = [2022, 2023, 2024, 2025]

train_X_list = []
train_y_list = []

val_X_list = []
val_y_list = []

test_X_list = []
test_y_list = []

for year in years:

    npz_path = os.path.join(
        base_dir,
        f"{year}_extended",
        f"drifter_{year}_extended_supervised_windows.npz"
    )

    d = load_year_npz(npz_path)

    train_X_list.append(d["X_train"])
    train_y_list.append(d["y_train"])

    val_X_list.append(d["X_val"])
    val_y_list.append(d["y_val"])

    test_X_list.append(d["X_test"])
    test_y_list.append(d["y_test"])

    feature_cols = d["feature_cols"]

X_train = np.concatenate(train_X_list, axis=0)
y_train = np.concatenate(train_y_list, axis=0)

X_val = np.concatenate(val_X_list, axis=0)
y_val = np.concatenate(val_y_list, axis=0)

X_test = np.concatenate(test_X_list, axis=0)
y_test = np.concatenate(test_y_list, axis=0)

print(X_train.shape)
print(y_train.shape)


(727755, 8, 21)
(727755, 2)


## Feature Selection

In [ ]:

selected_features = [
    "latitude",
    "longitude",
    "ve",
    "vn",
    "speed",
    "direction",
    "accel_east",
    "accel_north"
]

feature_cols = list(feature_cols)

feature_idx = [
    feature_cols.index(f)
    for f in selected_features
]

X_train = X_train[:, :, feature_idx]
X_val = X_val[:, :, feature_idx]
X_test = X_test[:, :, feature_idx]

##################################################
# Build REAL Multi-Step Supervised Windows
##################################################

HISTORY = 8
FUTURE = 4

selected_features = [
    "latitude",
    "longitude",
    "ve",
    "vn",
    "speed",
    "direction",
    "accel_east",
    "accel_north"
]

feature_cols = list(feature_cols)

feature_idx = [
    feature_cols.index(f)
    for f in selected_features
]

LAT_IDX = selected_features.index(
    "latitude"
)

LON_IDX = selected_features.index(
    "longitude"
)

METERS_PER_DEG = 111000


##################################################
# Build Windows
##################################################

def build_real_multistep_windows(

    X_raw
):

    X_out = []

    Y_out = []

    for i in range(

        HISTORY,

        len(X_raw) - FUTURE
    ):

        ##################################################
        # history
        ##################################################

        hist = X_raw[
            i-HISTORY:i,
            feature_idx
        ]

        ##################################################
        # current position
        ##################################################

        lat0 = X_raw[
            i-1,
            feature_cols.index("latitude")
        ]

        lon0 = X_raw[
            i-1,
            feature_cols.index("longitude")
        ]

        lat0_rad = np.deg2rad(lat0)

        ##################################################
        # future trajectory
        ##################################################

        future_steps = []

        for k in range(1, FUTURE+1):

            lat_f = X_raw[
                i+k-1,
                feature_cols.index("latitude")
            ]

            lon_f = X_raw[
                i+k-1,
                feature_cols.index("longitude")
            ]

            ##################################################
            # displacement
            ##################################################

            delta_lat = lat_f - lat0

            delta_lon = lon_f - lon0

            north_m = (
                delta_lat
                *
                METERS_PER_DEG
            )

            east_m = (
                delta_lon
                *
                METERS_PER_DEG
                *
                np.cos(lat0_rad)
            )

            future_steps.append(
                [north_m, east_m]
            )

        future_steps = np.array(
            future_steps
        )

        X_out.append(hist)

        Y_out.append(future_steps)

    return (

        np.array(X_out),

        np.array(Y_out)
    )


## Multi-Step Target Construction

In [ ]:

FUTURE_STEPS = 4

DT_SECONDS = 6 * 3600

METERS_PER_DEG = 111000

VE_IDX = selected_features.index("ve")
VN_IDX = selected_features.index("vn")
LAT_IDX = selected_features.index("latitude")

def build_multistep_targets(X, y):

    lat_now = X[:, -1, LAT_IDX]

    lat_rad = np.deg2rad(lat_now)

    delta_lat = y[:, 0]
    delta_lon = y[:, 1]

    final_north_m = (
        delta_lat * METERS_PER_DEG
    )

    final_east_m = (
        delta_lon
        *
        METERS_PER_DEG
        *
        np.cos(lat_rad)
    )

    targets = []

    for step in range(1, FUTURE_STEPS + 1):

        alpha = step / FUTURE_STEPS

        north_step = final_north_m * alpha

        east_step = final_east_m * alpha

        step_target = np.stack(
            [north_step, east_step],
            axis=1
        )

        targets.append(step_target)

    targets = np.stack(
        targets,
        axis=1
    )

    return targets

y_train_m = build_multistep_targets(
    X_train,
    y_train
)

y_val_m = build_multistep_targets(
    X_val,
    y_val
)

y_test_m = build_multistep_targets(
    X_test,
    y_test
)


## Advection Baseline

In [ ]:

def compute_multistep_advection(X):

    ve = X[:, -1, VE_IDX]

    vn = X[:, -1, VN_IDX]

    outputs = []

    for step in range(1, FUTURE_STEPS + 1):

        dt = step * DT_SECONDS

        east_m = ve * dt

        north_m = vn * dt

        out = np.stack(
            [north_m, east_m],
            axis=1
        )

        outputs.append(out)

    outputs = np.stack(
        outputs,
        axis=1
    )

    return outputs

adv_train = compute_multistep_advection(
    X_train
)

adv_val = compute_multistep_advection(
    X_val
)

adv_test = compute_multistep_advection(
    X_test
)


## Residual Normalization

In [ ]:

residual_train = y_train_m - adv_train
residual_val = y_val_m - adv_val
residual_test = y_test_m - adv_test

residual_mean = residual_train.mean(
    axis=(0,1)
)

residual_std = residual_train.std(
    axis=(0,1)
)

residual_std = np.where(
    residual_std < 1e-6,
    1.0,
    residual_std
)

residual_train_norm = (
    residual_train - residual_mean
) / residual_std

residual_val_norm = (
    residual_val - residual_mean
) / residual_std

residual_test_norm = (
    residual_test - residual_mean
) / residual_std


## Dataset

In [ ]:

class DrifterDataset(Dataset):

    def __init__(
        self,
        X,
        residual_norm,
        adv
    ):

        self.X = torch.tensor(
            X,
            dtype=torch.float32
        )

        self.y = torch.tensor(
            residual_norm,
            dtype=torch.float32
        )

        self.adv = torch.tensor(
            adv,
            dtype=torch.float32
        )

    def __len__(self):

        return len(self.X)

    def __getitem__(self, idx):

        return (
            self.X[idx],
            self.y[idx],
            self.adv[idx]
        )


## DataLoaders

In [ ]:

train_dataset = DrifterDataset(
    X_train,
    residual_train_norm,
    adv_train
)

val_dataset = DrifterDataset(
    X_val,
    residual_val_norm,
    adv_val
)

test_dataset = DrifterDataset(
    X_test,
    residual_test_norm,
    adv_test
)

train_loader = DataLoader(
    train_dataset,
    batch_size=64,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=64
)

test_loader = DataLoader(
    test_dataset,
    batch_size=64
)


## Positional Encoding

In [ ]:

class PositionalEncoding(nn.Module):

    def __init__(
        self,
        d_model,
        max_len=32
    ):

        super().__init__()

        pe = torch.zeros(max_len, d_model)

        position = torch.arange(
            0,
            max_len
        ).unsqueeze(1)

        div_term = torch.exp(
            torch.arange(
                0,
                d_model,
                2
            ) *
            (-np.log(10000.0) / d_model)
        )

        pe[:, 0::2] = torch.sin(
            position * div_term
        )

        pe[:, 1::2] = torch.cos(
            position * div_term
        )

        self.register_buffer(
            "pe",
            pe.unsqueeze(0)
        )

    def forward(self, x):

        return x + self.pe[:, :x.size(1)]


## Attention Transformer

In [ ]:

class TransformerEncoder(nn.Module):

    def __init__(
        self,
        input_dim,
        d_model=128,
        nhead=4,
        num_layers=2
    ):

        super().__init__()

        self.input_proj = nn.Linear(
            input_dim,
            d_model
        )

        self.pe = PositionalEncoding(d_model)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model,
            nhead=nhead,
            batch_first=True
        )

        self.encoder = nn.TransformerEncoder(
            encoder_layer,
            num_layers=num_layers
        )

        self.attn_pool = nn.Sequential(

            nn.Linear(d_model, d_model),

            nn.Tanh(),

            nn.Linear(d_model, 1)
        )

    def forward(self, x):

        x = self.input_proj(x)

        x = self.pe(x)

        h = self.encoder(x)

        scores = self.attn_pool(h)

        weights = torch.softmax(
            scores,
            dim=1
        )

        pooled = torch.sum(
            h * weights,
            dim=1
        )

        return pooled


## Neural ODE

In [ ]:

class ODEFunc(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(hidden_dim, hidden_dim),

            nn.Tanh(),

            nn.Linear(hidden_dim, hidden_dim)
        )

    def forward(self, t, z):

        return self.net(z)


class ODEBlock(nn.Module):

    def __init__(self, hidden_dim):

        super().__init__()

        self.func = ODEFunc(hidden_dim)

    def forward(self, z0):

        t = torch.tensor(
            [0,1,2,3],
            dtype=torch.float32,
            device=z0.device
        )

        z_t = odeint(
            self.func,
            z0,
            t,
            method='rk4'
        )

        z_t = z_t.permute(1,0,2)

        return z_t


## Full Model

In [ ]:

class TransformerODEModel(nn.Module):

    def __init__(self, input_dim):

        super().__init__()

        self.encoder = TransformerEncoder(
            input_dim=input_dim
        )

        self.ode = ODEBlock(128)

        self.decoder = nn.Sequential(

            nn.Linear(128,64),

            nn.ReLU(),

            nn.Linear(64,2)
        )

    def forward(self, x):

        z0 = self.encoder(x)

        z_t = self.ode(z0)

        outputs = []

        for i in range(FUTURE_STEPS):

            step_out = self.decoder(
                z_t[:,i]
            )

            outputs.append(step_out)

        outputs = torch.stack(
            outputs,
            dim=1
        )

        return outputs


## Training Setup

In [ ]:

device = torch.device(
    "cuda" if torch.cuda.is_available()
    else "cpu"
)

model = TransformerODEModel(
    input_dim=len(selected_features)
).to(device)

optimizer = optim.AdamW(
    model.parameters(),
    lr=1e-4,
    weight_decay=1e-6
)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',
    factor=0.5,
    patience=3
)

criterion = nn.SmoothL1Loss()

residual_mean_tensor = torch.tensor(
    residual_mean,
    dtype=torch.float32
).to(device)

residual_std_tensor = torch.tensor(
    residual_std,
    dtype=torch.float32
).to(device)


## Checkpoint Utils

In [ ]:

def find_latest_checkpoint(
    checkpoint_dir="checkpoints"
):

    if not os.path.exists(checkpoint_dir):

        return None

    checkpoints = [
        f for f in os.listdir(checkpoint_dir)
        if f.startswith("model_")
        and f.endswith(".pt")
    ]

    if not checkpoints:

        return None

    checkpoints.sort(
        key=lambda x: int(
            x.split('_')[1].split('.')[0]
        )
    )

    return os.path.join(
        checkpoint_dir,
        checkpoints[-1]
    )


## Validation Function

In [ ]:

def evaluate(model, loader):

    model.eval()

    total_loss_eval = 0

    with torch.no_grad():

        for Xb, yb, advb in loader:

            Xb = Xb.to(device)

            yb = yb.to(device)

            residual_norm_pred = model(Xb)

            loss = criterion(
                residual_norm_pred,
                yb
            )

            total_loss_eval += loss.item()

    return total_loss_eval / len(loader)


## Training Loop

In [ ]:

os.makedirs(
    "checkpoints",
    exist_ok=True
)

latest_checkpoint_path = find_latest_checkpoint()

start_epoch_idx = 0

if latest_checkpoint_path:

    checkpoint = torch.load(
        latest_checkpoint_path
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

    optimizer.load_state_dict(
        checkpoint["optimizer_state_dict"]
    )

    start_epoch_idx = checkpoint["epoch"]

save_every = 10

EPOCHS = 100

for current_epoch_idx in range(
    start_epoch_idx,
    EPOCHS
):

    model.train()

    train_loss = 0

    for Xb, yb, advb in tqdm(train_loader):

        Xb = Xb.to(device)

        yb = yb.to(device)

        optimizer.zero_grad()

        residual_norm_pred = model(Xb)

        loss = criterion(
            residual_norm_pred,
            yb
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        optimizer.step()

        train_loss += loss.item()

    train_loss /= len(train_loader)

    val_loss = evaluate(
        model,
        val_loader
    )

    scheduler.step(val_loss)

    current_lr = optimizer.param_groups[0]['lr']

    print(
        f"Epoch {current_epoch_idx+1} | "
        f"Train Loss: {train_loss:.6f} | "
        f"Val Loss: {val_loss:.6f} | "
        f"LR: {current_lr:.2e}"
    )

    if (current_epoch_idx + 1) % save_every == 0:

        torch.save({

            "epoch":
                current_epoch_idx + 1,

            "model_state_dict":
                model.state_dict(),

            "optimizer_state_dict":
                optimizer.state_dict(),

            "train_loss":
                train_loss,

            "val_loss":
                val_loss

        },

        f"checkpoints/model_{current_epoch_idx + 1}.pt")


100%|██████████| 11372/11372 [03:40<00:00, 51.55it/s]


Epoch 61 | Train Loss: 0.005076 | Val Loss: 0.005694 | LR: 1.56e-06


100%|██████████| 11372/11372 [03:39<00:00, 51.76it/s]


Epoch 62 | Train Loss: 0.005070 | Val Loss: 0.005678 | LR: 1.56e-06


100%|██████████| 11372/11372 [03:40<00:00, 51.48it/s]


...

Epoch 80 | Train Loss: 0.005047 | Val Loss: 0.005693 | LR: 4.88e-08


100%|██████████| 11372/11372 [03:39<00:00, 51.76it/s]


Epoch 81 | Train Loss: 0.005043 | Val Loss: 0.005692 | LR: 4.88e-08


 42%|████▏     | 4824/11372 [01:32<02:06, 51.88it/s]


KeyboardInterrupt: 

## Evaluation

In [ ]:

model.eval()

latest_checkpoint_path = find_latest_checkpoint()
# latest_checkpoint_path = "/content/checkpoints/model_50.pt"

if latest_checkpoint_path:

    checkpoint = torch.load(
        latest_checkpoint_path
    )

    model.load_state_dict(
        checkpoint["model_state_dict"]
    )

preds = []

with torch.no_grad():

    for Xb, _, advb in test_loader:

        Xb = Xb.to(device)

        advb = advb.to(device)

        residual_norm_pred = model(Xb)

        residual_physical = (

            residual_norm_pred
            *
            residual_std_tensor

            +

            residual_mean_tensor
        )

        pred = advb + residual_physical

        pred = pred[:, -1]

        preds.append(
            pred.cpu().numpy()
        )

preds = np.concatenate(preds)

targets = y_test_m[:, -1]

METERS_PER_DEG = 111000

LAT_IDX = selected_features.index(
    "latitude"
)

LON_IDX = selected_features.index(
    "longitude"
)

lat_now = X_test[:, -1, LAT_IDX]

lat_rad = np.deg2rad(lat_now)

pred_delta_lat = (
    preds[:, 0]
    /
    METERS_PER_DEG
)

pred_delta_lon = (
    preds[:, 1]
    /
    (
        METERS_PER_DEG
        *
        np.cos(lat_rad)
    )
)

target_delta_lat = (
    targets[:, 0]
    /
    METERS_PER_DEG
)

target_delta_lon = (
    targets[:, 1]
    /
    (
        METERS_PER_DEG
        *
        np.cos(lat_rad)
    )
)
##################################################
# RMSE Metrics
##################################################

rmse_lat = np.sqrt(
    mean_squared_error(
        target_delta_lat,
        pred_delta_lat
    )
)

rmse_lon = np.sqrt(
    mean_squared_error(
        target_delta_lon,
        pred_delta_lon
    )
)

print(
    "RMSE Latitude:",
    rmse_lat
)

print(
    "RMSE Longitude:",
    rmse_lon
)


##################################################
# Recover Absolute Positions
##################################################

current_lat = X_test[
    :,
    -1,
    LAT_IDX
]

current_lon = X_test[
    :,
    -1,
    LON_IDX
]

pred_future_lat = (
    current_lat
    +
    pred_delta_lat
)

pred_future_lon = (
    current_lon
    +
    pred_delta_lon
)

target_future_lat = (
    current_lat
    +
    target_delta_lat
)

target_future_lon = (
    current_lon
    +
    target_delta_lon
)


##################################################
# Haversine Distance
##################################################

def haversine_km(

    lat1,
    lon1,
    lat2,
    lon2
):

    R = 6371.0

    lat1 = np.deg2rad(lat1)
    lon1 = np.deg2rad(lon1)

    lat2 = np.deg2rad(lat2)
    lon2 = np.deg2rad(lon2)

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = (

        np.sin(dlat / 2) ** 2

        +

        np.cos(lat1)
        *
        np.cos(lat2)
        *
        np.sin(dlon / 2) ** 2
    )

    c = 2 * np.arcsin(
        np.sqrt(a)
    )

    return R * c


##################################################
# Geo Distance Metrics
##################################################

geo_errors = haversine_km(

    target_future_lat,
    target_future_lon,

    pred_future_lat,
    pred_future_lon
)

mean_geo = np.mean(
    geo_errors
)

median_geo = np.median(
    geo_errors
)

p90_geo = np.percentile(
    geo_errors,
    90
)

print(
    "Mean geo (km):",
    mean_geo
)

print(
    "Median geo (km):",
    median_geo
)

print(
    "P90 geo (km):",
    p90_geo
)


##################################################
# Summary Table
##################################################

results_df = pd.DataFrame({

    "Metric": [

        "RMSE Latitude",

        "RMSE Longitude",

        "Mean geo (km)",

        "Median geo (km)",

        "P90 geo (km)"
    ],

    "Value": [

        rmse_lat,

        rmse_lon,

        mean_geo,

        median_geo,

        p90_geo
    ]
})

print(results_df)

RMSE Latitude: 0.05908096131230832
RMSE Longitude: 0.7424824981681788
Mean geo (km): 8.570863
Median geo (km): 5.232087
P90 geo (km): 14.959751
            Metric      Value
0    RMSE Latitude   0.059081
1   RMSE Longitude   0.742482
2    Mean geo (km)   8.570863
3  Median geo (km)   5.232087
4     P90 geo (km)  14.959751
